In [1]:
import os
%pwd

'/home/tuhin/bangla-political-memes-classification/research'

In [2]:
import os
if os.path.basename(os.getcwd()) == 'research':
    os.chdir("../")

In [3]:
%pwd

'/home/tuhin/bangla-political-memes-classification'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class NeuralNetworkModelUsingTextConfig:
    root_dir: Path
    train_features_csv: Path
    test_features_csv: Path
    model_path: Path
    epochs: int
    batch_size: int
    learning_rate: float
    hidden_dim: int

In [5]:
from memeClassifier.constants import *
from memeClassifier.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_neural_network_model_using_text_config(self) -> NeuralNetworkModelUsingTextConfig:
        config = self.config.neural_network_model_using_text
        params = self.params.NeuralNetworkModelUsingText

        create_directories([config.root_dir])

        neural_network_config = NeuralNetworkModelUsingTextConfig(
            root_dir=Path(config.root_dir),
            train_features_csv=Path(config.train_features_csv),
            test_features_csv=Path(config.test_features_csv),
            model_path=Path(config.model_path),
            epochs=params.EPOCHS,
            batch_size=params.BATCH_SIZE,
            learning_rate=params.LEARNING_RATE,
            hidden_dim=params.HIDDEN_DIM
        )

        return neural_network_config

In [7]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from memeClassifier import logger

class SimpleNN(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(SimpleNN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        return self.network(x)

class NeuralNetworkModelUsingText:
    def __init__(self, config: NeuralNetworkModelUsingTextConfig):
        self.config = config
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def initiate_model_training(self):
        logger.info("Loading feature datasets")
        train_df = pd.read_csv(self.config.train_features_csv)
        
        X = train_df[['political_specific_count', 'political_specific_ratio']].values
        y = train_df['Label'].map({'Political': 1, 'NonPolitical': 0}).values
        
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        logger.info(f"Training set size: {X_train.shape[0]}")
        logger.info(f"Validation set size: {X_val.shape[0]}")
        
        # Convert to PyTorch tensors
        X_train_tensor = torch.FloatTensor(X_train).to(self.device)
        y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1).to(self.device)
        X_val_tensor = torch.FloatTensor(X_val).to(self.device)
        y_val_tensor = torch.FloatTensor(y_val).unsqueeze(1).to(self.device)
        
        # Create DataLoader
        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
        train_loader = DataLoader(train_dataset, batch_size=self.config.batch_size, shuffle=True)
        
        # Initialize model, loss function, optimizer
        model = SimpleNN(input_dim=X_train.shape[1], hidden_dim=self.config.hidden_dim).to(self.device)
        criterion = nn.BCELoss()
        optimizer = optim.Adam(model.parameters(), lr=self.config.learning_rate)
        
        logger.info(f"Training Neural Network for {self.config.epochs} epochs...")
        for epoch in range(self.config.epochs):
            model.train()
            for batch_X, batch_y in train_loader:
                optimizer.zero_grad()
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
        
        # Evaluation
        model.eval()
        with torch.no_grad():
            train_outputs = model(X_train_tensor)
            train_preds = (train_outputs >= 0.5).float().cpu().numpy()
            
            val_outputs = model(X_val_tensor)
            val_preds = (val_outputs >= 0.5).float().cpu().numpy()
        
        train_accuracy = accuracy_score(y_train, train_preds)
        val_accuracy = accuracy_score(y_val, val_preds)
        
        logger.info(f"Training Accuracy: {train_accuracy:.4f}")
        logger.info(f"Validation Accuracy: {val_accuracy:.4f}")
        logger.info(f"Classification Report (Validation Set):\n{classification_report(y_val, val_preds, target_names=['NonPolitical', 'Political'])}")
        
        torch.save(model.state_dict(), self.config.model_path)
        logger.info(f"PyTorch Neural Network Model saved to {self.config.model_path}")

In [8]:
try:
    config = ConfigurationManager()
    neural_network_config = config.get_neural_network_model_using_text_config()
    classifier = NeuralNetworkModelUsingText(config=neural_network_config)
    classifier.initiate_model_training()
except Exception as e:
    raise e

[2026-06-27 02:30:15,984: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-06-27 02:30:15,986: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-27 02:30:15,987: INFO: common: Directory created at: artifacts]
[2026-06-27 02:30:15,987: INFO: common: Directory created at: artifacts/neural_network_model_using_text]
[2026-06-27 02:30:15,988: INFO: 4000206215: Loading feature datasets]
[2026-06-27 02:30:15,993: INFO: 4000206215: Training set size: 156]
[2026-06-27 02:30:15,994: INFO: 4000206215: Validation set size: 39]
[2026-06-27 02:30:16,919: INFO: 4000206215: Training Neural Network for 50 epochs...]
[2026-06-27 02:30:17,456: INFO: 4000206215: Training Accuracy: 0.7821]
[2026-06-27 02:30:17,457: INFO: 4000206215: Validation Accuracy: 0.7949]
[2026-06-27 02:30:17,462: INFO: 4000206215: Classification Report (Validation Set):
              precision    recall  f1-score   support

NonPolitical       0.78      1.00      0.88        29
   Political     